# Project: A Personal Assistant with Subagents

This project builds a personal assistant that coordinates two specialists with very different
responsibilities:

- A **calendar agent** that handles scheduling, availability checking, and event management.
- An **email agent** that manages communication, drafts messages, and sends notifications.

A **supervisor** agent coordinates both, deciding which one to call for a given request and
combining their results into one coherent answer. We'll also add **human-in-the-loop review**
so a person can approve, edit, or reject sensitive actions (like an outbound email) before
they actually happen.

```mermaid
graph TD
    U([User request]) --> SV[Supervisor agent]
    SV -->|schedule_event tool| CA[Calendar agent]
    SV -->|manage_email tool| EA[Email agent]
    CA -->|create_calendar_event| C1[(Calendar API)]
    CA -->|get_available_time_slots| C1
    EA -->|send_email| C2[(Email API)]
    CA -.response.-> SV
    EA -.response.-> SV
    SV --> R([Combined answer])
```

## Why a supervisor, instead of one agent with every tool?

Give one agent direct access to both calendar and email APIs, and it has to choose from
similar-looking tools, understand each API's exact format, and juggle two unrelated domains in
one prompt. Splitting the work into two focused specialists, each with its own tools and
prompt, keeps each one simple -- and makes each one independently testable.

## Setup

Copy `.env.example` to `.env` and set `OPENAI_API_KEY=sk-...` before running this notebook.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set. Copy .env.example to .env and add your key.")

from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-5-mini")
print("Model ready.")


## 1. Define the low-level tools

These are the tools that actually touch an API. In a real system they'd call Google Calendar,
Outlook, SendGrid, or similar. Here they're stubs that return a plausible result, so the whole
notebook runs without any external service.


In [ ]:
from langchain.tools import tool

@tool
def create_calendar_event(
    title: str,
    start_time: str,       # ISO format: "2024-01-15T14:00:00"
    end_time: str,         # ISO format: "2024-01-15T15:00:00"
    attendees: list[str],  # email addresses
    location: str = ""
) -> str:
    """Create a calendar event. Requires exact ISO datetime format."""
    return f"Event created: {title} from {start_time} to {end_time} with {len(attendees)} attendees"


@tool
def send_email(
    to: list[str],
    subject: str,
    body: str,
    cc: list[str] = []
) -> str:
    """Send an email via email API. Requires properly formatted addresses."""
    return f"Email sent to {', '.join(to)} - Subject: {subject}"


@tool
def get_available_time_slots(
    attendees: list[str],
    date: str,  # ISO format: "2024-01-15"
    duration_minutes: int
) -> list[str]:
    """Check calendar availability for given attendees on a specific date."""
    return ["09:00", "14:00", "16:00"]

print("Tools defined.")


## 2. Create the two specialist agents

Each agent gets only the tools and the prompt it needs. The calendar agent never sees
`send_email`; the email agent never sees `create_calendar_event`.


In [ ]:
from datetime import date
from langchain.agents import create_agent

CALENDAR_AGENT_PROMPT = (
    f"Today's date is {date.today().isoformat()}. "
    "You are a calendar scheduling assistant. "
    "Parse natural language scheduling requests (e.g., 'next Tuesday at 2pm') "
    "into proper ISO datetime formats. "
    "Use get_available_time_slots to check availability when needed. "
    "If there is no suitable time slot, stop and confirm unavailability in your response. "
    "Use create_calendar_event to schedule events. "
    "Always confirm what was scheduled in your final response."
)

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
)

EMAIL_AGENT_PROMPT = (
    "You are an email assistant. "
    "Compose professional emails based on natural language requests. "
    "Extract recipient information and craft appropriate subject lines and body text. "
    "Use send_email to send the message. "
    "Always confirm what was sent in your final response."
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
)

print("Both specialist agents created.")


Try the calendar agent on its own first, before wiring it into anything bigger:


In [ ]:
result = calendar_agent.invoke({
    "messages": [{"role": "user", "content": "Schedule a team meeting next Tuesday at 2pm for 1 hour"}]
})
for msg in result["messages"]:
    msg.pretty_print()


And the email agent on its own:


In [ ]:
result = email_agent.invoke({
    "messages": [{"role": "user", "content": "Send the design team a reminder about reviewing the new mockups"}]
})
for msg in result["messages"]:
    msg.pretty_print()


Each specialist works well on its own. Now we wrap each one as a tool so a supervisor can
call them.

## 3. Wrap each specialist as a tool

This is the key architectural step. The supervisor will see a high-level tool like
`schedule_event`, never the low-level `create_calendar_event` API tool.

```mermaid
graph LR
    SV[Supervisor] -->|"schedule_event('meeting next Tuesday at 2pm')"| T1[schedule_event tool]
    T1 -->|invokes| CA[calendar_agent]
    CA -->|final message only| T1
    T1 -->|return string| SV
```


In [ ]:
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language.

    Use this when the user wants to create, modify, or check calendar appointments.
    Handles date/time parsing, availability checking, and event creation.

    Input: Natural language scheduling request (e.g., 'meeting with design team
    next Tuesday at 2pm')
    """
    result = calendar_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].content


@tool
def manage_email(request: str) -> str:
    """Send emails using natural language.

    Use this when the user wants to send notifications, reminders, or any email
    communication. Handles recipient extraction, subject generation, and email
    composition.

    Input: Natural language email request (e.g., 'send them a reminder about
    the meeting')
    """
    result = email_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].content

print("Tool wrappers defined -- only each sub-agent's final answer flows back up.")


Note that we return only the sub-agent's final response, not its intermediate tool calls. The
supervisor doesn't need to see *how* the calendar agent found an open slot -- only the result.

## 4. Create the supervisor

The supervisor only sees the two high-level tools above, and routes at the domain level.


In [ ]:
SUPERVISOR_PROMPT = (
    "You are a helpful personal assistant. "
    "You can schedule calendar events and send emails. "
    "Break down user requests into appropriate tool calls and coordinate the results. "
    "When a request involves multiple actions, use multiple tools in sequence or in parallel as appropriate."
)

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
)
print("Supervisor created.")


## 5. Use the supervisor

### Example 1: a simple, single-domain request


In [ ]:
result = supervisor_agent.invoke({
    "messages": [{"role": "user", "content": "Schedule a team standup for tomorrow at 9am"}]
})
for msg in result["messages"]:
    msg.pretty_print()


The supervisor recognized this as a calendar task and called `schedule_event`, which delegated
to the calendar agent.

### Example 2: a complex, multi-domain request


In [ ]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)
result = supervisor_agent.invoke({"messages": [{"role": "user", "content": query}]})
for msg in result["messages"]:
    msg.pretty_print()


This time the model issued **two tool calls in the same turn** -- `schedule_event` and
`manage_email` -- and the runtime executed them. Each sub-agent completed its own task
independently, and the supervisor combined both results into one answer.

```mermaid
graph TD
    U([Schedule a meeting and<br/>send a reminder]) --> SV[Supervisor]
    SV --> T1[schedule_event]
    SV --> T2[manage_email]
    T1 --> CA[Calendar agent]
    T2 --> EA[Email agent]
    CA -.-> SV
    EA -.-> SV
    SV --> R([One combined response])
```

### Understanding the three layers

| Layer | Contains | Responsibility |
|---|---|---|
| Bottom | `create_calendar_event`, `send_email`, `get_available_time_slots` | Rigid, exact-format API tools |
| Middle | `calendar_agent`, `email_agent` | Translate natural language into structured calls, and back |
| Top | `supervisor_agent` | Routes to high-level capabilities, synthesizes results |

Each layer can be tested and improved on its own, and adding a new domain (say, a restaurant
reservation agent) means adding a new specialist and a new wrapper tool -- the supervisor and
the existing specialists don't need to change.

## 6. Add human-in-the-loop review

Before any of this touches a real calendar or sends a real email, it's worth pausing for human
approval on sensitive actions. LangChain has built-in middleware for exactly this.

We configure `create_calendar_event` and `send_email` to interrupt execution, and add a
**checkpointer** to the top-level supervisor (required to pause and resume a run).


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"create_calendar_event": True},
            description_prefix="Calendar event pending approval",
        ),
    ],
)

email_agent = create_agent(
    model,
    tools=[send_email],
    system_prompt=EMAIL_AGENT_PROMPT,
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email": True},
            description_prefix="Outbound email pending approval",
        ),
    ],
)

# Re-define the wrapper tools so they close over the NEW calendar_agent/email_agent above.
@tool
def schedule_event(request: str) -> str:
    """Schedule calendar events using natural language."""
    result = calendar_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].content

@tool
def manage_email(request: str) -> str:
    """Send emails using natural language."""
    result = email_agent.invoke({"messages": [{"role": "user", "content": request}]})
    return result["messages"][-1].content

supervisor_agent = create_agent(
    model,
    tools=[schedule_event, manage_email],
    system_prompt=SUPERVISOR_PROMPT,
    checkpointer=InMemorySaver(),
)
print("Supervisor rebuilt with human-in-the-loop review and a checkpointer.")


Run the same multi-domain query again. This time, instead of completing both actions, the
agent should **pause** and wait for approval.


In [ ]:
query = (
    "Schedule a meeting with the design team next Tuesday at 2pm for 1 hour, "
    "and send them an email reminder about reviewing the new mockups."
)
config = {"configurable": {"thread_id": "demo-thread-1"}}

result = supervisor_agent.invoke({"messages": [{"role": "user", "content": query}]}, config)

interrupt_info = result.get("__interrupt__")
if interrupt_info:
    print(f"INTERRUPTED -- {len(interrupt_info)} action(s) waiting for approval:\n")
    for interrupt_ in interrupt_info:
        for request in interrupt_.value["action_requests"]:
            print(f"ID: {interrupt_.id}")
            print(f"{request['description']}\n")
else:
    print("No interrupt -- the model didn't call a HITL-guarded tool this time. Try re-running.")


If you saw two interrupts above, we can now decide what happens to each one: approve, edit, or
reject. Here we'll approve the calendar event as-is, but edit the email's subject line before
it goes out.


In [ ]:
from langgraph.types import Command

resume = {}
if interrupt_info:
    for interrupt_ in interrupt_info:
        action = interrupt_.value["action_requests"][0]
        if action["name"] == "send_email":
            edited_action = action.copy()
            edited_action["args"] = dict(edited_action["args"])
            edited_action["args"]["subject"] = "Mockups reminder"
            resume[interrupt_.id] = {"decisions": [{"type": "edit", "edited_action": edited_action}]}
        else:
            resume[interrupt_.id] = {"decisions": [{"type": "approve"}]}

    result = supervisor_agent.invoke(Command(resume=resume), config)
    for msg in result["messages"]:
        msg.pretty_print()
else:
    print("Nothing to resume -- no interrupt was captured above.")


The run proceeds with our decisions: the calendar event goes through unchanged, and the email
is sent with the edited subject line. This is the same `Command(resume=...)` mechanism used
anywhere LangChain pauses for human input -- see the human-in-the-loop guide for the full set
of decision types (`approve`, `edit`, `reject`).

## 7. Advanced: controlling information flow

By default, a sub-agent only receives the request string the supervisor passes it -- not the
full conversation. Sometimes you want more context to flow in, or less to flow back out.

These next two examples are independent of the human-in-the-loop demo above, so we use a
fresh, plain calendar agent here (not the HITL-wrapped `calendar_agent` from section 6).

### Pass more context into a sub-agent


In [ ]:
# A plain calendar agent for these two illustrations -- deliberately not the HITL-wrapped
# `calendar_agent` from section 6, since calling that one outside a checkpointed supervisor
# run would try to interrupt with nowhere to resume from.
plain_calendar_agent = create_agent(
    model,
    tools=[create_calendar_event, get_available_time_slots],
    system_prompt=CALENDAR_AGENT_PROMPT,
)
print("Plain calendar agent ready for the examples below.")


In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def schedule_event_with_context(request: str, runtime: ToolRuntime) -> str:
    """Schedule calendar events using natural language, with full conversation context."""
    original_user_message = next(
        message for message in runtime.state["messages"]
        if message.type == "human"
    )
    prompt = (
        "You are assisting with the following user inquiry:\n\n"
        f"{original_user_message.text}\n\n"
        "You are tasked with the following sub-request:\n\n"
        f"{request}"
    )
    result = plain_calendar_agent.invoke({"messages": [{"role": "user", "content": prompt}]})
    return result["messages"][-1].content

print("Context-forwarding tool defined -- useful for requests like 'same time as yesterday'.")


This lets the calendar agent see the *entire* conversation, not just its own sub-request --
useful for resolving references like "schedule it for the same time tomorrow."

### Control what the supervisor receives back

The reverse direction matters too: a sub-agent's final message should contain everything the
supervisor needs, because the supervisor only sees that one string.


In [ ]:
import json

@tool
def schedule_event_structured(request: str) -> str:
    """Schedule calendar events; returns structured data instead of a plain confirmation."""
    result = plain_calendar_agent.invoke({"messages": [{"role": "user", "content": request}]})
    # Option 1 (used throughout this notebook): return just the confirmation message.
    # Option 2: return structured data instead, for a supervisor that needs to act on fields:
    return json.dumps({
        "status": "success",
        "summary": result["messages"][-1].content,
    })

print(schedule_event_structured.func("Schedule a demo for Thursday at 11am"))


**A common failure mode worth remembering:** a sub-agent that calls its tools correctly but
doesn't restate the results in its final message -- make sure every specialist's prompt says
to confirm what happened in the final response.

## Key takeaways

| Point | Why it matters |
|---|---|
| Wrap sub-agents as tools | This is the entire Subagents mechanism -- nothing more exotic than that |
| Give each specialist a narrow prompt and a small toolset | Keeps each layer testable and easy to improve independently |
| Return only the final message from a sub-agent | The supervisor doesn't need (and shouldn't see) intermediate tool calls |
| Add `HumanInTheLoopMiddleware` for sensitive tools | Pauses before anything irreversible happens, with `approve`/`edit`/`reject` |
| A checkpointer is required on the top-level agent for HITL | Without it, a paused run has nowhere to resume from |

**When to use this pattern:** multiple distinct domains, each with its own tools or complex
logic, where you want centralized control and sub-agents don't need to talk to the user
directly. For simpler cases with just a few tools, a single agent is enough. When agents need
to converse directly with users, look at the Handoffs pattern instead (next notebook).
